In [23]:
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np 
orders = pd.read_csv("../data/processed/shopee_orders_thailand.csv")
order_items = pd.read_csv("../data/processed/shopee_order_items_thailand.csv")
customers = pd.read_csv("../data/processed/shopee_customers_thailand.csv")
products = pd.read_csv("../data/processed/shopee_products_thailand.csv")
product_campaigns = pd.read_csv("../data/processed/shopee_product_campaign_thailand.csv")
sellers = pd.read_csv("../data/processed/shopee_sellers_thailand.csv")
campaigns = pd.read_csv("../data/processed/shopee_campaigns_thailand.csv")
shipments = pd.read_csv("../data/processed/shopee_shipments_thailand.csv")


In [26]:
#KHÁCH HÀNG NÀO MANG LẠI DOANH THU NHIỀU NHẤT
customers["dob"] = pd.to_datetime(customers["dob"])

today = pd.to_datetime("today")

customers["age"] = today.year - customers["dob"].dt.year - (
    (today.month < customers["dob"].dt.month) |
    (
        (today.month == customers["dob"].dt.month)
        & (today.day < customers["dob"].dt.day)
    )
)

customers["age_group"] = pd.cut(
    customers["age"],
    bins=[0,18,25,35,45,60,100],
    labels=["<18","18-25","26-35","36-45","46-60","60+"]
)
customer_orders = orders.merge(
    customers[["customer_id", "age_group"]],
    on="customer_id",
    how="left"
)

age_revenue = (
    customer_orders.groupby("age_group")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

age_revenue

age_group
46-60    1.007681e+09
26-35    9.596421e+08
60+      8.228924e+08
36-45    7.811514e+08
18-25    5.463440e+08
<18      7.166869e+06
Name: total_amount, dtype: float64

In [8]:
merged_df.columns

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'unit_price_after_discount', 'line_total', 'discount_percent',
       'commission_amount', 'maintenance_amount', 'shipping_fee_item',
       'estimated_delivery_start', 'estimated_delivery_end', 'item_status',
       'is_campaign', 'product_campaign_id', 'seller_id', 'category',
       'product_name', 'maintenance_rate', 'commission_rate', 'weight',
       'created_at'],
      dtype='str')

In [9]:
#DANH MỤC SẢN PHẨM NÀO MANG LẠI DOANH THU NHIỀU NHẤT
category_revenue = (
    merged_df.groupby("category")["line_total"]
    .sum()
    .sort_values(ascending=False)
)

category_revenue

category
Home           2.540428e+09
Electronics    1.380483e+09
Fashion        1.162734e+08
Groceries      2.040555e+07
Beauty         1.654468e+07
Name: line_total, dtype: float64

In [13]:
#LOẠI SHOP NÀO HOẠT ĐỘNG TỐT NHẤT
products_sellers = products.merge(
    sellers[["seller_id", "seller_type"]],
    on="seller_id",
    how="left"
)
seller_revenue = (
    merged_df.merge(
        sellers[["seller_id", "seller_type"]],
        on="seller_id",
        how="left"
    )
)

seller_type_revenue = (
    seller_revenue.groupby("seller_type")["line_total"]
    .sum()
    .sort_values(ascending=False)
)

seller_type_revenue

seller_type
Company       3.150234e+09
Individual    9.239008e+08
Name: line_total, dtype: float64

In [14]:
#TỈNH/ THÀNH PHỐ NÀO CÓ NHIỀU SHOP BÁN HÀNG TRÊN SÀN NHẤT?
sellers["province"].value_counts().head(10)

province
Samut Prakan     58
Pathum Thani     55
Nakhon Pathom    47
Bangkok          20
Samut Sakhon     19
Nonthaburi        1
Name: count, dtype: int64

In [16]:
campaigns.columns

Index(['campaign_id', 'campaign_name', 'start_date', 'end_date',
       'campaign_type'],
      dtype='str')

In [17]:
product_campaigns.columns

Index(['product_campaign_id', 'product_id', 'campaign_id', 'discount_percent'], dtype='str')

In [20]:
#CAMPAIGN CÓ MỨC GIẢM GIÁ TRUNG BÌNH CAO NHẤT
campaign_analysis = product_campaigns.merge(
    campaigns,
    on="campaign_id",
    how="left"
)

campaign_result = (
    campaign_analysis.groupby("campaign_name")["discount_percent"]
    .mean()
    .sort_values(ascending=False)
)

campaign_result


campaign_name
Songkran Sale    8.222355
10.10 Sale       7.876984
9.9 Sale         7.568956
11.11 Sale       7.270932
12.12 Sale       6.874318
Name: discount_percent, dtype: float64

## Business Insights

1. INSIGHT VỀ KHÁCH HÀNG
- Nhóm tuổi từ 46-60 tuổi tạo ra doanh thu cao nhất với hơn 1 tỷ THB, cho thấy đây là nhóm khách hàng có sức mua mạnh và giá trị đơn hàng cao.
- Nhóm khách hàng 26-35 tuổi đứng thứ hai về doanh thu, là phân khúc khách hàng quan trọng cần được duy trì và phát triển.
- Khách hàng dưới 18 tuổi đóng góp rất ít doanh thu, cho thấy đây không phải là khách hàng quan trọng của nền tảng.
# ĐỀ XUẤT
- Tập trung các chương trình khuyến mãi và chăm sóc khách hàng cho nhóm 26-60 tuổi.
- Xây dựng các chương trình khách hàng thân thiết dành cho nhóm khách hàng có giá trị mua sắm cao.
2. INSIGHT VỀ SẢN PHẨM
- Danh mục Home là danh mục mang lại doanh thu cao nhất với khoảng 2,54 tỷ THB, chiếm tỷ trọng lớn trong tổng doanh thu.
- Danh mục Electronics đứng thứ hai với khoảng 1,38 tỷ THB.
- Các danh mục Beauty và Groceries có doanh thu thấp hơn đáng kể so với các nhóm còn lại.
# ĐỀ XUẤT
- Ưu tiên đầu tư quảng bá cho các danh mục Home và Electronics.
- Xem xét triển khai các chiến dịch kích cầu cho Beauty và Groceries để cải thiện doanh thu.
3. INSIGHT VỀ NGƯỜI BÁN
- Các shop thuộc nhóm Company tạo ra khoảng 3,15 tỷ THB doanh thu, cao gấp nhiều lần so với nhóm Individual.
- Điều này cho thấy các doanh nghiệp có khả năng cung cấp sản phẩm đa dạng và tạo doanh thu ổn định hơn so với người bán cá nhân.
# ĐỀ XUẤT
- Tăng cường thu hút các doanh nghiệp tham gia nền tảng.
- Hỗ trợ các shop cá nhân nâng cao năng lực bán hàng thông qua các chương trình đào tạo và khuyến khích.
4. INSIGHT VỀ ĐỊA LÝ
- Samut Prakan, Pathum Thani và Nakhon Pathom là những địa phương có số lượng người bán cao nhất.
- Mặc dù Bangkok là trung tâm kinh tế lớn nhưng số lượng seller trong dữ liệu thấp hơn nhiều so với các tỉnh lân cận.
# ĐỀ XUẤT
- Tăng cường mở rộng mạng lưới người bán tại các khu vực còn ít nhà bán hàng.
- Nghiên cứu nguyên nhân tập trung seller tại các tỉnh vệ tinh để tối ưu chiến lược phát triển thị trường.
5. INSIGHT VỀ CHIẾN DỊCH MARKETING
- Songkran Sale là chiến dịch có mức giảm giá trung bình cao nhất (8,22%).
- Các chiến dịch lớn như 10.10 Sale, 9.9 Sale, 11.11 Sale và 12.12 Sale đều có mức giảm giá cao và duy trì sức hấp dẫn đối với khách hàng.
# ĐỀ XUẤT 
- Tiếp tục đầu tư vào các chiến dịch theo mùa và các sự kiện mua sắm lớn.
- Tối ưu mức giảm giá để cân bằng giữa doanh thu và lợi nhuận.